In [1]:
import warnings
import numpy as np
import pandas as pd
import features as fe

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
SEED = 42
np.random.seed(SEED)

In [2]:
train, test, events = fe.load_data("data")
print("train", train.shape, "test", test.shape, "events", events.shape)
print("bot share:", round(train.target.mean(), 4))
print("train windows:", train.window_start_ts.min().date(), "-", train.window_start_ts.max().date())
print("test windows:", test.window_start_ts.min().date(), "-", test.window_start_ts.max().date())
print("window length:", (train.window_end_ts - train.window_start_ts).unique())
print("train/test cookie overlap:", len(set(train.cookie_id) & set(test.cookie_id)))

train (11091, 5) test (4909, 4) events (328905, 14)
bot share: 0.0811
train windows: 2026-04-06 - 2026-04-19
test windows: 2026-04-20 - 2026-04-26
window length: <TimedeltaArray>
['1 days']
Length: 1, dtype: timedelta64[ns]
train/test cookie overlap: 0


In [3]:
print("duplicate rows:", events.duplicated().sum())
print("sorted by time:", events.event_ts.is_monotonic_increasing)
print(events.platform.value_counts().to_dict())
events.isna().mean().round(3).to_frame("nan_share").T

duplicate rows: 4863
sorted by time: False


{'desktop': 46866, 'WEB': 46476, 'Web': 46365, 'web': 45951, 'ANDROID': 42462, 'Android': 42254, 'android': 42159, 'iphone': 4103, 'IOS': 4100, 'iOS': 4091, 'ios': 4078}


,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y
nan_share,0.0,0.0,0.0,0.0,0.0,0.0,0.348,0.1,0.072,0.407,0.695,0.695,0.67,0.67


In [4]:
meta_all = pd.concat([train, test])
m = events.merge(meta_all[["cookie_id", "window_start_ts", "window_end_ts"]], on="cookie_id")
m["position"] = np.select([m.event_ts < m.window_start_ts, m.event_ts >= m.window_end_ts],
                          ["before", "after"], "inside")
m["split"] = np.where(m.cookie_id.isin(train.cookie_id), "train", "test")
display(pd.crosstab(m.event_name, m.position))
after = m[m.position == "after"].groupby("cookie_id").size()
print("events after window per cookie (train):",
      train.assign(n=train.cookie_id.map(after).fillna(0)).groupby("target").n.mean().round(2).to_dict())
print("events after window in test:", (m[m.split == "test"].position == "after").sum())

position,after,inside
event_name,,
captcha_shown,7928,0
contact_chat_open,577,5548
contact_message_sent,258,2823
contact_phone_show,1028,10288
favorite_add,1753,17296
item_view,12731,108086
login,606,5661
photo_swipe,3433,33084
search_results_view,10604,89798


events after window per cookie (train): {0: 2.08, 1: 21.77}
events after window in test: 0


In [5]:
inside = m[(m.position == "inside") & (m.split == "train")].merge(train[["cookie_id", "target"]], on="cookie_id")
display(pd.crosstab(inside.event_name, inside.target, normalize="columns").round(3))
dup = events.duplicated(keep=False)
print("duplicate share by target:",
      events.assign(d=dup).merge(train, on="cookie_id").groupby("target").d.mean().round(4).to_dict())

target,0,1
event_name,,
contact_chat_open,0.020,0.014
contact_message_sent,0.010,0.008
contact_phone_show,0.037,0.026
favorite_add,0.066,0.027
item_view,0.365,0.436
login,0.021,0.008
photo_swipe,0.123,0.070
search_results_view,0.305,0.352
seller_page_view,0.053,0.059


duplicate share by target: {0: 0.0297, 1: 0.0285}
